In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import ast
import os

#Load news.tsv
news_columns = [
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities"
]

news_df = pd.read_csv("/kaggle/input/datasets/teshanlakruwan/mind-small-train/news.tsv",
    sep="\t",
    header=None,
    names=news_columns
)

print("News shape:", news_df.shape)
news_df.head()

#Load behaviors.tsv 

behaviors_columns = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions"
]

behaviors_df = pd.read_csv("/kaggle/input/datasets/teshanlakruwan/mind-small-train/behaviors.tsv",
    sep="\t",
    header=None,
    names=behaviors_columns
)

print("Behaviors shape:", behaviors_df.shape)
behaviors_df.head()

#Inspect caegories in the dataset
print("Unique categories:")
print(news_df["category"].unique())

print("\nCategory counts:")
print(news_df["category"].value_counts())

news_df["category_lower"] = news_df["category"].str.lower().str.strip()
news_df["subcategory_lower"] = news_df["subcategory"].str.lower().str.strip()

#Maping MIND categories to compatible with IAB content taxonomy 3.0 categories
mind_to_iab = {
    # Core categories
    "sports": "Sports",
    "finance": "Business_Finance",
    "autos": "Automotive",
    "travel": "Travel",
    "health": "Health",
    "lifestyle": "Lifestyle",
    "foodanddrink": "Food_Drink",

    # News-related
    "news": "General_News",
    "weather": "General_News",
    "middleeast": "General_News",
    "northamerica": "General_News",

    # Entertainment group
    "entertainment": "Entertainment",
    "tv": "Entertainment",
    "movies": "Entertainment",
    "music": "Entertainment",
    "video": "Entertainment",

    # Edge case
    "kids": "Lifestyle"   # or "Entertainment" (either is fine)
}

news_df["category_lower"] = news_df["category"].str.lower().str.strip()

def map_to_iab(category):
    return mind_to_iab.get(category, "Other")

news_df["iab_category"] = news_df["category_lower"].apply(map_to_iab)

print(news_df["iab_category"].value_counts())

#Remove Other type of categories if exists
news_df = news_df[news_df["iab_category"] != "Other"].copy()

#Validate If we got balanced datarecords for categories
print(news_df["iab_category"].value_counts())

#Get the news ids in behaviours to a list
def parse_history(history):
    if pd.isna(history) or history == "":
        return []
    return history.split()

behaviors_df["history_list"] = behaviors_df["history"].apply(parse_history)
behaviors_df["history_count"] = behaviors_df["history_list"].apply(len)

behaviors_df.head()


# Parse Impressions column data
def parse_impressions(impressions):
    result = []
    if pd.isna(impressions):
        return result
    
    for item in impressions.split():
        news_id, clicked = item.split("-")
        result.append((news_id, int(clicked)))
    return result

behaviors_df["impression_items"] = behaviors_df["impressions"].apply(parse_impressions)

behaviors_df.head()

#Expand Impressions
rows = []

for _, row in behaviors_df.iterrows():
    for news_id, clicked in row["impression_items"]:
        rows.append({
            "impression_id": row["impression_id"],
            "user_id": row["user_id"],
            "time": row["time"],
            "history_list": row["history_list"],
            "history_count": row["history_count"],
            "candidate_news_id": news_id,
            "clicked": clicked
        })

expanded_df = pd.DataFrame(rows)

print("Expanded shape:", expanded_df.shape)
expanded_df.head()


#Join refined two tables
master_df = expanded_df.merge(
    news_df,
    left_on="candidate_news_id",
    right_on="news_id",
    how="inner"
)

print("Master shape:", master_df.shape)
master_df.head()

#Add simple behavioural features
master_df["history_unique_count"] = master_df["history_list"].apply(lambda x: len(set(x)))

master_df["current_in_history"] = master_df.apply(
    lambda row: 1 if row["candidate_news_id"] in row["history_list"] else 0,
    axis=1
)

#Save Master dataset
master_df = master_df.sample(n=200000, random_state=42)
master_df.to_csv("/kaggle/working/master_train_dataset.csv", index=False)

News shape: (51282, 8)
Behaviors shape: (156965, 5)
Unique categories:
['lifestyle' 'health' 'news' 'sports' 'weather' 'entertainment' 'autos'
 'travel' 'foodanddrink' 'tv' 'finance' 'movies' 'video' 'music' 'kids'
 'middleeast' 'northamerica']

Category counts:
category
news             15774
sports           14510
finance           3107
foodanddrink      2551
lifestyle         2479
travel            2350
video             2068
weather           2048
health            1885
autos             1639
tv                 889
music              769
movies             606
entertainment      587
kids                17
middleeast           2
northamerica         1
Name: count, dtype: int64
iab_category
General_News        17825
Sports              14510
Entertainment        4919
Business_Finance     3107
Food_Drink           2551
Lifestyle            2496
Travel               2350
Health               1885
Automotive           1639
Name: count, dtype: int64
iab_category
General_News        17825

In [2]:
#Prepare Test dataset from Mind_small_dev dataset
#Load news.tsv
dev_news_df = pd.read_csv(
    "/kaggle/input/datasets/teshanlakruwan/mind-small-dev/news.tsv",
    sep="\t",
    header=None,
    names=news_columns
)

#Load behaviours.tsv
dev_behaviors_df = pd.read_csv(
    "/kaggle/input/datasets/teshanlakruwan/mind-small-dev/behaviors.tsv",
    sep="\t",
    header=None,
    names=behaviors_columns
)

dev_news_df = dev_news_df[[
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract"
]]

dev_news_df["category_lower"] = dev_news_df["category"].str.lower().str.strip()

dev_news_df["iab_category"] = dev_news_df["category_lower"].apply(map_to_iab)

dev_news_df = dev_news_df[dev_news_df["iab_category"] != "Other"].copy()

print(dev_news_df["iab_category"].value_counts())

#history
dev_behaviors_df["history_list"] = dev_behaviors_df["history"].apply(parse_history)
dev_behaviors_df["history_count"] = dev_behaviors_df["history_list"].apply(len)

#Impressions
dev_behaviors_df["impression_items"] = dev_behaviors_df["impressions"].apply(parse_impressions)

#expand Impressions
rows = []

for _, row in dev_behaviors_df.iterrows():
    for news_id, clicked in row["impression_items"]:
        rows.append({
            "impression_id": row["impression_id"],
            "user_id": row["user_id"],
            "time": row["time"],
            "history_list": row["history_list"],
            "history_count": row["history_count"],
            "candidate_news_id": news_id,
            "clicked": clicked
        })

dev_expanded_df = pd.DataFrame(rows)

print(dev_expanded_df.shape)
dev_expanded_df.head()

#join with news test data
master_dev_df = dev_expanded_df.merge(
    dev_news_df,
    left_on="candidate_news_id",
    right_on="news_id",
    how="inner"
)

#Combine behavioural features

print(master_dev_df.shape)
master_dev_df.head()

master_dev_df["history_unique_count"] = master_dev_df["history_list"].apply(lambda x: len(set(x)))

master_dev_df["current_in_history"] = master_dev_df.apply(
    lambda row: 1 if row["candidate_news_id"] in row["history_list"] else 0,
    axis=1
)

#save data
master_dev_df = master_dev_df.sample(n=50000, random_state=42)
master_dev_df.to_csv("/kaggle/working/master_dev_dataset.csv", index=False)

iab_category
General_News        14507
Sports              11760
Entertainment        4144
Business_Finance     2563
Food_Drink           2248
Lifestyle            2143
Travel               1845
Health               1715
Automotive           1490
Name: count, dtype: int64
(2740998, 7)
(2740998, 14)


In [3]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer


#Fill missing textfields
for df in [master_df, master_dev_df]:
    df["title"] = df["title"].fillna("")
    df["abstract"] = df["abstract"].fillna("")
    df["subcategory"] = df["subcategory"].fillna("")

    # Combine text fields into one column
    df["text"] = (
        df["title"].astype(str) + " " +
        df["abstract"].astype(str) + " " +
        df["subcategory"].astype(str)
    ).str.strip()

print(master_df[["text", "iab_category"]].head())
print(master_dev_df[["text", "iab_category"]].head())

# Labels
y_train = master_df["iab_category"]
y_dev = master_dev_df["iab_category"]

# Text input
X_train_text_raw = master_df["text"]
X_dev_text_raw = master_dev_df["text"]

# Behaviour features
behaviour_cols = [
    "history_count",
    "history_unique_count",
    "current_in_history"
]

X_train_behaviour = master_df[behaviour_cols].copy()
X_dev_behaviour = master_dev_df[behaviour_cols].copy()

print("Train labels shape:", y_train.shape)
print("Dev labels shape:", y_dev.shape)
print("Train behaviour shape:", X_train_behaviour.shape)
print("Dev behaviour shape:", X_dev_behaviour.shape)


#TF-IDF Vectorization

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5
)

X_train_text = tfidf.fit_transform(X_train_text_raw)
X_dev_text = tfidf.transform(X_dev_text_raw)

print("TF-IDF train shape:", X_train_text.shape)
print("TF-IDF dev shape:", X_dev_text.shape)

#Encode labels


label_encoder = LabelEncoder()

y_train_enc = label_encoder.fit_transform(y_train)
y_dev_enc = label_encoder.transform(y_dev)

print("Classes:", label_encoder.classes_)


                                                      text      iab_category
3874367  Carrie Underwood rocks CMAs and several gorgeo...         Lifestyle
4422490  Kodak Black Sentenced to Over 3 Years in Priso...     Entertainment
147236   'Priceless' finds that turned out to be worthl...  Business_Finance
5023576  The highlights and lowlights from the world of...     Entertainment
5455306  Groom Makes The Most Heartfelt Vows To His 9-Y...         Lifestyle
                                                      text   iab_category
1371415  How much turkey do you need to buy per person?...     Food_Drink
559975   At NATO summit, Trump to stress US allies' def...   General_News
2488017  Wrecked Toyota Supra Launch Edition Didn't Eve...     Automotive
1258873  New Ant Species Discovered in Ant Expert's Bac...  Entertainment
247757   Months-Long 60 Swarm Construction To Conclude ...     Automotive
Train labels shape: (200000,)
Dev labels shape: (50000,)
Train behaviour shape: (200000, 3)
De

In [4]:
# Test text only prediction test
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

text_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

text_model.fit(X_train_text, y_train_enc)

y_pred_text = text_model.predict(X_dev_text)

acc = accuracy_score(y_dev_enc, y_pred_text)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_dev_enc, y_pred_text, average="weighted"
)

print("=== Text Only Results ===")
print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1-score :", f1)

print("\nDetailed Report:")
print(classification_report(
    y_dev_enc,
    y_pred_text,
    target_names=label_encoder.classes_
))

=== Text Only Results ===
Accuracy : 0.9596
Precision: 0.9612984747585197
Recall   : 0.9596
F1-score : 0.9590425023890318

Detailed Report:
                  precision    recall  f1-score   support

      Automotive       0.91      0.96      0.94      2024
Business_Finance       1.00      1.00      1.00      4096
   Entertainment       0.98      0.99      0.98      9779
      Food_Drink       0.95      1.00      0.97      3863
    General_News       0.92      1.00      0.96     12533
          Health       1.00      0.94      0.97      2478
       Lifestyle       0.98      0.92      0.95      6505
          Sports       0.97      0.91      0.94      6410
          Travel       1.00      0.77      0.87      2312

        accuracy                           0.96     50000
       macro avg       0.97      0.94      0.95     50000
    weighted avg       0.96      0.96      0.96     50000



In [5]:
#behaviours only prdiction test
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_behaviour_scaled = scaler.fit_transform(X_train_behaviour)
X_dev_behaviour_scaled = scaler.transform(X_dev_behaviour)

behaviour_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

behaviour_model.fit(X_train_behaviour_scaled, y_train_enc)

y_pred_behaviour = behaviour_model.predict(X_dev_behaviour_scaled)

acc = accuracy_score(y_dev_enc, y_pred_behaviour)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_dev_enc, y_pred_behaviour, average="weighted"
)

print("=== Behaviour Only Results ===")
print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1-score :", f1)

print("\nDetailed Report:")
print(classification_report(
    y_dev_enc,
    y_pred_behaviour,
    target_names=label_encoder.classes_
))

=== Behaviour Only Results ===
Accuracy : 0.15558
Precision: 0.12790090020422168
Recall   : 0.15558
F1-score : 0.08048990427955524

Detailed Report:
                  precision    recall  f1-score   support

      Automotive       0.00      0.00      0.00      2024
Business_Finance       0.00      0.00      0.00      4096
   Entertainment       0.19      0.63      0.30      9779
      Food_Drink       0.08      0.29      0.12      3863
    General_News       0.27      0.00      0.00     12533
          Health       0.00      0.00      0.00      2478
       Lifestyle       0.00      0.00      0.00      6505
          Sports       0.13      0.07      0.09      6410
          Travel       0.00      0.00      0.00      2312

        accuracy                           0.16     50000
       macro avg       0.07      0.11      0.06     50000
    weighted avg       0.13      0.16      0.08     50000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [6]:
#Textual + Behavioural Prdiction test
from scipy.sparse import hstack, csr_matrix

X_train_combined = hstack([
    X_train_text,
    csr_matrix(X_train_behaviour_scaled)
])

X_dev_combined = hstack([
    X_dev_text,
    csr_matrix(X_dev_behaviour_scaled)
])

combined_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

combined_model.fit(X_train_combined, y_train_enc)

y_pred_combined = combined_model.predict(X_dev_combined)

acc = accuracy_score(y_dev_enc, y_pred_combined)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_dev_enc, y_pred_combined, average="weighted"
)

print("=== Text + Behaviour Results ===")
print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1-score :", f1)

print("\nDetailed Report:")
print(classification_report(
    y_dev_enc,
    y_pred_combined,
    target_names=label_encoder.classes_
))

=== Text + Behaviour Results ===
Accuracy : 0.9574
Precision: 0.9589181240691995
Recall   : 0.9574
F1-score : 0.9568494662080478

Detailed Report:
                  precision    recall  f1-score   support

      Automotive       0.98      0.95      0.96      2024
Business_Finance       1.00      1.00      1.00      4096
   Entertainment       0.97      0.97      0.97      9779
      Food_Drink       0.95      1.00      0.97      3863
    General_News       0.92      1.00      0.96     12533
          Health       1.00      0.96      0.98      2478
       Lifestyle       0.98      0.92      0.95      6505
          Sports       0.95      0.91      0.93      6410
          Travel       1.00      0.77      0.87      2312

        accuracy                           0.96     50000
       macro avg       0.97      0.94      0.95     50000
    weighted avg       0.96      0.96      0.96     50000



In [7]:
# Compare 3 approches and evaluate it
results = []

def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted"
    )
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1_score": f1
    })

evaluate_model("Text Only", y_dev_enc, y_pred_text)
evaluate_model("Behaviour Only", y_dev_enc, y_pred_behaviour)
evaluate_model("Text + Behaviour", y_dev_enc, y_pred_combined)

results_df = pd.DataFrame(results)
print(results_df)

              Model  Accuracy  Precision   Recall  F1_score
0         Text Only   0.95960   0.961298  0.95960  0.959043
1    Behaviour Only   0.15558   0.127901  0.15558  0.080490
2  Text + Behaviour   0.95740   0.958918  0.95740  0.956849


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [8]:
import joblib

joblib.dump(tfidf, "tfidf_vectorizer.pkl")
joblib.dump(text_model, "stage1_model.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("Saved Stage 1 artifacts")

Saved Stage 1 artifacts


In [9]:
# Predict on TRAIN
train_preds = text_model.predict(X_train_text)
train_probs = text_model.predict_proba(X_train_text)

# Predict on DEV
dev_preds = text_model.predict(X_dev_text)
dev_probs = text_model.predict_proba(X_dev_text)

# Confidence = max probability
train_conf = train_probs.max(axis=1)
dev_conf = dev_probs.max(axis=1)

# Convert back to category names
train_labels = label_encoder.inverse_transform(train_preds)
dev_labels = label_encoder.inverse_transform(dev_preds)

# Add to dataset
master_df["predicted_category"] = train_labels
master_df["stage1_confidence"] = train_conf

master_dev_df["predicted_category"] = dev_labels
master_dev_df["stage1_confidence"] = dev_conf

# Save
master_df.to_csv("train_with_stage1_outputs.csv", index=False)
master_dev_df.to_csv("dev_with_stage1_outputs.csv", index=False)

print("Stage 1 outputs saved")

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

behaviour_cols = [
    "history_count",
    "history_unique_count",
    "current_in_history"
]

# Fit on train
master_df[behaviour_cols] = scaler.fit_transform(master_df[behaviour_cols])

# Apply to dev
master_dev_df[behaviour_cols] = scaler.transform(master_dev_df[behaviour_cols])

# Create behaviour score
master_df["behaviour_score"] = (
    0.4 * master_df["history_count"] +
    0.3 * master_df["history_unique_count"] +
    0.3 * master_df["current_in_history"]
)

master_dev_df["behaviour_score"] = (
    0.4 * master_dev_df["history_count"] +
    0.3 * master_dev_df["history_unique_count"] +
    0.3 * master_dev_df["current_in_history"]
)

# Save
master_df.to_csv("train_with_behaviour_score.csv", index=False)
master_dev_df.to_csv("dev_with_behaviour_score.csv", index=False)

print("Behaviour score added")

Stage 1 outputs saved
Behaviour score added


In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


# 1. Load ads

ads_df = pd.read_csv("/kaggle/input/datasets/teshanlakruwan/ad-pool/ads_pool.csv")

# Clean text
ads_df["ad_text"] = ads_df["ad_text"].fillna("").astype(str)

# Make sure required columns exist
required_ad_cols = ["ad_id", "ad_text", "category", "type"]
for col in required_ad_cols:
    if col not in ads_df.columns:
        raise ValueError(f"Missing required ad column: {col}")

required_page_cols = [
    "predicted_category",
    "stage1_confidence",
    "behaviour_score",
    "clicked"
]
for col in required_page_cols:
    if col not in master_df.columns:
        raise ValueError(f"Missing required page column: {col}")


# 2. Reset index Very important: matrix rows use position 0..n-1

master_df = master_df.reset_index(drop=True)
ads_df = ads_df.reset_index(drop=True)


# 3. Transform ad text using SAME TF-IDF

ad_vectors = tfidf.transform(ads_df["ad_text"])

# Page vectors already exist from Stage 1
page_vectors = X_train_text


# 4. Candidate sampling settings Avoid page x all ads explosion
same_cat_n = 3
other_cat_n = 2
generic_n = 1

pairs = []

# Pre-filter generic ads once
generic_ads = ads_df[ads_df["type"].str.lower() == "generic"]


# 5. Build page-ad pairs

for i, row in master_df.iterrows():
    page_vec = page_vectors[i]
    page_cat = row["predicted_category"]

    # Candidate ad groups
    same_cat_ads = ads_df[ads_df["category"] == page_cat]
    other_cat_ads = ads_df[ads_df["category"] != page_cat]

    # Sample small candidate sets
    sampled_same = (
        same_cat_ads.sample(n=min(same_cat_n, len(same_cat_ads)), random_state=42)
        if len(same_cat_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
    )

    sampled_other = (
        other_cat_ads.sample(n=min(other_cat_n, len(other_cat_ads)), random_state=42)
        if len(other_cat_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
    )

    sampled_generic = (
        generic_ads.sample(n=min(generic_n, len(generic_ads)), random_state=42)
        if len(generic_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
    )

    # Merge and remove duplicates
    candidate_ads = pd.concat(
        [sampled_same, sampled_other, sampled_generic],
        ignore_index=True
    ).drop_duplicates(subset=["ad_id"])

    # Compare current page with each candidate ad
    for _, ad_row in candidate_ads.iterrows():
        ad_idx = ads_df.index[ads_df["ad_id"] == ad_row["ad_id"]][0]
        ad_vec = ad_vectors[ad_idx]

        # Text similarity
        sim = cosine_similarity(page_vec, ad_vec)[0][0]

        # Category match
        category_match = int(page_cat == ad_row["category"])

        # Ad type
        ad_type_targeted = int(str(ad_row["type"]).lower() == "targeted")

        # Proxy label
        # Positive only if page was clicked AND category matches
        label = int((row["clicked"] == 1) and (category_match == 1))

        pairs.append({
            "page_id": i,
            "candidate_news_id": row["candidate_news_id"] if "candidate_news_id" in master_df.columns else i,
            "ad_id": ad_row["ad_id"],
            "page_category": page_cat,
            "ad_category": ad_row["category"],
            "stage1_confidence": row["stage1_confidence"],
            "behaviour_score": row["behaviour_score"],
            "text_similarity": sim,
            "category_match": category_match,
            "ad_type_targeted": ad_type_targeted,
            "label": label
        })


# 6. Save pair dataset

pair_df = pd.DataFrame(pairs)

pair_df.to_csv("/kaggle/working/stage2_train_pairs.csv", index=False)

print("Stage 2 dataset created:", pair_df.shape)
print(pair_df.head())

Stage 2 dataset created: (1200000, 11)
   page_id candidate_news_id     ad_id page_category    ad_category  \
0        0            N23805  AD_00616     Lifestyle      Lifestyle   
1        0            N23805  AD_00030     Lifestyle      Lifestyle   
2        0            N23805  AD_00432     Lifestyle      Lifestyle   
3        0            N23805  AD_00794     Lifestyle         Health   
4        0            N23805  AD_00803     Lifestyle  Entertainment   

   stage1_confidence  behaviour_score  text_similarity  category_match  \
0           0.952536         0.001266              0.0               1   
1           0.952536         0.001266              0.0               1   
2           0.952536         0.001266              0.0               1   
3           0.952536         0.001266              0.0               0   
4           0.952536         0.001266              0.0               0   

   ad_type_targeted  label  
0                 1      0  
1                 1      0  
2 

In [11]:
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    roc_auc_score,
    confusion_matrix
)
from sklearn.model_selection import train_test_split

# =========================
# 1. Load Stage 2 pair dataset
# =========================
pair_df = pd.read_csv("/kaggle/working/stage2_train_pairs.csv")

print("Pair dataset shape:", pair_df.shape)
print(pair_df.head())

# =========================
# 2. Define features and label
# =========================
feature_cols = [
    "stage1_confidence",
    "behaviour_score",
    "text_similarity",
    "category_match",
    "ad_type_targeted"
]

label_col = "label"

# Clean
pair_df[feature_cols] = pair_df[feature_cols].fillna(0)
pair_df[label_col] = pd.to_numeric(pair_df[label_col], errors="coerce").fillna(0).astype(int)

X = pair_df[feature_cols]
y = pair_df[label_col]

print("\nLabel distribution:")
print(y.value_counts(normalize=True))

# =========================
# 3. Train / validation split
# =========================
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Validation shape:", X_val.shape)

# =========================
# 4. Train model
# class_weight='balanced' helps if labels are imbalanced
# =========================
stage2_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

stage2_model.fit(X_train, y_train)

# =========================
# 5. Evaluate
# =========================
y_pred = stage2_model.predict(X_val)
y_prob = stage2_model.predict_proba(X_val)[:, 1]

acc = accuracy_score(y_val, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_val,
    y_pred,
    average="weighted",
    zero_division=0
)

print("\n=== Stage 2 Suitability Model Results ===")
print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1-score :", f1)

if len(np.unique(y_val)) > 1:
    auc = roc_auc_score(y_val, y_prob)
    print("ROC-AUC  :", auc)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

print("\nDetailed Report:")
print(classification_report(y_val, y_pred, zero_division=0))

# =========================
# 6. Feature importance
# =========================
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": stage2_model.coef_[0]
}).sort_values(by="coefficient", ascending=False)

print("\nFeature Coefficients:")
print(coef_df)

# =========================
# 7. Save model
# =========================
joblib.dump(stage2_model, "/kaggle/working/stage2_suitability_model.pkl")

print("\nSaved: /kaggle/working/stage2_suitability_model.pkl")

print(y.value_counts(normalize=True))

Pair dataset shape: (1200000, 11)
   page_id candidate_news_id     ad_id page_category    ad_category  \
0        0            N23805  AD_00616     Lifestyle      Lifestyle   
1        0            N23805  AD_00030     Lifestyle      Lifestyle   
2        0            N23805  AD_00432     Lifestyle      Lifestyle   
3        0            N23805  AD_00794     Lifestyle         Health   
4        0            N23805  AD_00803     Lifestyle  Entertainment   

   stage1_confidence  behaviour_score  text_similarity  category_match  \
0           0.952536         0.001266              0.0               1   
1           0.952536         0.001266              0.0               1   
2           0.952536         0.001266              0.0               1   
3           0.952536         0.001266              0.0               0   
4           0.952536         0.001266              0.0               0   

   ad_type_targeted  label  
0                 1      0  
1                 1      0  
2      

In [12]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics.pairwise import cosine_similarity

# =========================
# 1. Load saved models
# =========================
tfidf = joblib.load("/kaggle/working/tfidf_vectorizer.pkl")
stage1_model = joblib.load("/kaggle/working/stage1_model.pkl")
label_encoder = joblib.load("/kaggle/working/label_encoder.pkl")
stage2_model = joblib.load("/kaggle/working/stage2_suitability_model.pkl")

# =========================
# 2. Load dev pages + ads
# =========================
dev_df = pd.read_csv("/kaggle/working/dev_with_behaviour_score.csv")
ads_df = pd.read_csv("/kaggle/input/datasets/teshanlakruwan/ad-pool/ads_pool.csv")

ads_df["ad_text"] = ads_df["ad_text"].fillna("").astype(str)
ads_df = ads_df.reset_index(drop=True)

# =========================
# 3. Choose one test page
# Change index if needed
# =========================
page_idx = 0
page_row = dev_df.iloc[page_idx]

# Build page text if not already present
if "page_text" in dev_df.columns:
    page_text = str(page_row["page_text"])
elif "text" in dev_df.columns:
    page_text = str(page_row["text"])
else:
    page_text = (
        str(page_row.get("title", "")) + " " +
        str(page_row.get("abstract", "")) + " " +
        str(page_row.get("subcategory", ""))
    ).strip()

# =========================
# 4. Stage 1 prediction for this new page
# =========================
page_vec = tfidf.transform([page_text])

stage1_pred_enc = stage1_model.predict(page_vec)[0]
stage1_probs = stage1_model.predict_proba(page_vec)[0]

predicted_category = label_encoder.inverse_transform([stage1_pred_enc])[0]
stage1_confidence = stage1_probs.max()

# =========================
# 5. Behaviour score for this page
# Use saved value if already computed
# =========================
if "behaviour_score" in dev_df.columns:
    behaviour_score = float(page_row["behaviour_score"])
else:
    # fallback simple calculation if needed
    hc = float(page_row.get("history_count", 0))
    huc = float(page_row.get("history_unique_count", 0))
    cih = float(page_row.get("current_in_history", 0))
    behaviour_score = 0.4 * hc + 0.3 * huc + 0.3 * cih

# =========================
# 6. Candidate ad selection
# Same idea as training
# =========================
same_cat_n = 5
other_cat_n = 3
generic_n = 2

same_cat_ads = ads_df[ads_df["category"] == predicted_category]
other_cat_ads = ads_df[ads_df["category"] != predicted_category]
generic_ads = ads_df[ads_df["type"].str.lower() == "generic"]

sampled_same = (
    same_cat_ads.sample(n=min(same_cat_n, len(same_cat_ads)), random_state=42)
    if len(same_cat_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
)

sampled_other = (
    other_cat_ads.sample(n=min(other_cat_n, len(other_cat_ads)), random_state=42)
    if len(other_cat_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
)

sampled_generic = (
    generic_ads.sample(n=min(generic_n, len(generic_ads)), random_state=42)
    if len(generic_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
)

candidate_ads = pd.concat(
    [sampled_same, sampled_other, sampled_generic],
    ignore_index=True
).drop_duplicates(subset=["ad_id"]).reset_index(drop=True)

# =========================
# 7. Transform candidate ad text
# =========================
ad_vectors = tfidf.transform(candidate_ads["ad_text"])

# =========================
# 8. Build page-ad features and score
# =========================
rank_rows = []

for i, ad_row in candidate_ads.iterrows():
    ad_vec = ad_vectors[i]

    text_similarity = cosine_similarity(page_vec, ad_vec)[0][0]
    category_match = int(predicted_category == ad_row["category"])
    ad_type_targeted = int(str(ad_row["type"]).lower() == "targeted")

    feature_row = pd.DataFrame([{
        "stage1_confidence": stage1_confidence,
        "behaviour_score": behaviour_score,
        "text_similarity": text_similarity,
        "category_match": category_match,
        "ad_type_targeted": ad_type_targeted
    }])

    suitability_score = stage2_model.predict_proba(feature_row)[0][1]

    rank_rows.append({
        "ad_id": ad_row["ad_id"],
        "ad_text": ad_row["ad_text"],
        "ad_category": ad_row["category"],
        "ad_type": ad_row["type"],
        "stage1_confidence": stage1_confidence,
        "behaviour_score": behaviour_score,
        "text_similarity": text_similarity,
        "category_match": category_match,
        "ad_type_targeted": ad_type_targeted,
        "suitability_score": suitability_score
    })

ranked_ads = pd.DataFrame(rank_rows).sort_values(
    by="suitability_score",
    ascending=False
).reset_index(drop=True)

# =========================
# 9. Show results
# =========================
print("=== NEW PAGE ===")
print("Page index:", page_idx)
print("Predicted category:", predicted_category)
print("Stage 1 confidence:", stage1_confidence)
print("Behaviour score:", behaviour_score)
print()
print("Page text:")
print(page_text[:500])

print("\n=== TOP RANKED ADS ===")
print(ranked_ads[[
    "ad_id",
    "ad_category",
    "ad_type",
    "text_similarity",
    "category_match",
    "suitability_score",
    "ad_text"
]].head(10))

=== NEW PAGE ===
Page index: 0
Predicted category: Food_Drink
Stage 1 confidence: 0.9949928190451234
Behaviour score: 0.0088640749931072

Page text:
How much turkey do you need to buy per person? When hosting Thanksgiving dinner, there are a lot of questions regarding the menu that may come up. tipsandtricks

=== TOP RANKED ADS ===
      ad_id    ad_category   ad_type  text_similarity  category_match  \
0  AD_00343     Food_Drink  targeted         0.000000               1   
1  AD_00809     Food_Drink  targeted         0.000000               1   
2  AD_00394     Food_Drink  targeted         0.000000               1   
3  AD_00775     Food_Drink  targeted         0.000000               1   
4  AD_00844     Food_Drink  targeted         0.062029               1   
5   GEN_005        General   generic         0.000000               0   
6   GEN_002        General   generic         0.000000               0   
7  AD_00595      Lifestyle  targeted         0.000000               0   
8  AD_004

In [13]:
import pandas as pd
import numpy as np
from collections import Counter

# =========================
# 1. Load datasets
# =========================
master_df = pd.read_csv("/kaggle/working/train_with_behaviour_score.csv")
ads_df = pd.read_csv("/kaggle/input/datasets/teshanlakruwan/ad-pool/ads_pool.csv")

# =========================
# 2. Basic cleaning
# =========================
master_df = master_df.reset_index(drop=True)
ads_df = ads_df.reset_index(drop=True)

# Build page text if not already present
if "page_text" not in master_df.columns:
    if "text" in master_df.columns:
        master_df["page_text"] = master_df["text"].fillna("").astype(str)
    else:
        master_df["page_text"] = (
            master_df["title"].fillna("").astype(str) + " " +
            master_df["abstract"].fillna("").astype(str) + " " +
            master_df["subcategory"].fillna("").astype(str)
        ).str.strip()

ads_df["ad_text"] = ads_df["ad_text"].fillna("").astype(str)
ads_df["category"] = ads_df["category"].fillna("Unknown").astype(str)
ads_df["type"] = ads_df["type"].fillna("generic").astype(str)

# Safety checks
required_page_cols = [
    "predicted_category",
    "stage1_confidence",
    "behaviour_score",
    "clicked",
    "page_text"
]

for col in required_page_cols:
    if col not in master_df.columns:
        raise ValueError(f"Missing required page column: {col}")

required_ad_cols = ["ad_id", "ad_text", "category", "type"]
for col in required_ad_cols:
    if col not in ads_df.columns:
        raise ValueError(f"Missing required ad column: {col}")

# =========================
# 3. Better text similarity
# Jaccard overlap on token sets
# =========================
def tokenize_text(text):
    if pd.isna(text):
        return []
    return str(text).lower().split()

def simple_overlap_similarity(text1, text2):
    set1 = set(tokenize_text(text1))
    set2 = set(tokenize_text(text2))

    if len(set1) == 0 or len(set2) == 0:
        return 0.0

    return len(set1.intersection(set2)) / len(set1.union(set2))

# =========================
# 4. Candidate sampling settings
# =========================
same_cat_n = 3
other_cat_n = 2
generic_n = 1

pairs = []

generic_ads = ads_df[ads_df["type"].str.lower() == "generic"]

# =========================
# 5. Build page-ad training pairs
# =========================
for i, row in master_df.iterrows():
    page_cat = row["predicted_category"]
    page_text = row["page_text"]

    same_cat_ads = ads_df[ads_df["category"] == page_cat]
    other_cat_ads = ads_df[ads_df["category"] != page_cat]

    sampled_same = (
        same_cat_ads.sample(n=min(same_cat_n, len(same_cat_ads)), random_state=42)
        if len(same_cat_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
    )

    sampled_other = (
        other_cat_ads.sample(n=min(other_cat_n, len(other_cat_ads)), random_state=42)
        if len(other_cat_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
    )

    sampled_generic = (
        generic_ads.sample(n=min(generic_n, len(generic_ads)), random_state=42)
        if len(generic_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
    )

    candidate_ads = pd.concat(
        [sampled_same, sampled_other, sampled_generic],
        ignore_index=True
    ).drop_duplicates(subset=["ad_id"])

    for _, ad_row in candidate_ads.iterrows():
        ad_text = ad_row["ad_text"]

        # improved similarity
        text_similarity = simple_overlap_similarity(page_text, ad_text)

        category_match = int(page_cat == ad_row["category"])
        ad_type_targeted = int(str(ad_row["type"]).lower() == "targeted")

        # improved proxy label
        # positive if clicked and either category matches or text overlap is meaningful
        label = int(
            (row["clicked"] == 1) and
            (
                category_match == 1 or
                text_similarity > 0.05
            )
        )

        pairs.append({
            "page_id": i,
            "candidate_news_id": row["candidate_news_id"] if "candidate_news_id" in master_df.columns else i,
            "ad_id": ad_row["ad_id"],
            "page_text": page_text,
            "ad_text": ad_text,
            "page_category": page_cat,
            "ad_category": ad_row["category"],
            "stage1_confidence": float(row["stage1_confidence"]),
            "behaviour_score": float(row["behaviour_score"]),
            "text_similarity": float(text_similarity),
            "category_match": int(category_match),
            "ad_type_targeted": int(ad_type_targeted),
            "label": int(label)
        })

pair_df = pd.DataFrame(pairs)

pair_df.to_csv("/kaggle/working/stage2_train_pairs_fixed.csv", index=False)

print("Saved: /kaggle/working/stage2_train_pairs_fixed.csv")
print("Shape:", pair_df.shape)
print(pair_df.head())

print("\nLabel distribution:")
print(pair_df["label"].value_counts(normalize=True))

print("\nText similarity summary:")
print(pair_df["text_similarity"].describe())

print("\nZero similarity ratio:")
print((pair_df["text_similarity"] == 0).mean())

Saved: /kaggle/working/stage2_train_pairs_fixed.csv
Shape: (1200000, 13)
   page_id candidate_news_id     ad_id  \
0        0            N23805  AD_00616   
1        0            N23805  AD_00030   
2        0            N23805  AD_00432   
3        0            N23805  AD_00794   
4        0            N23805  AD_00803   

                                           page_text  \
0  Carrie Underwood rocks CMAs and several gorgeo...   
1  Carrie Underwood rocks CMAs and several gorgeo...   
2  Carrie Underwood rocks CMAs and several gorgeo...   
3  Carrie Underwood rocks CMAs and several gorgeo...   
4  Carrie Underwood rocks CMAs and several gorgeo...   

                                             ad_text page_category  \
0  +\ndigital\ntrends\neverything\nthat works\nwi...     Lifestyle   
1  vis\npromo codes\navailable\nsee codes\ncape s...     Lifestyle   
2  wizarding\nworld\n| harry potter\nbano 1\nhot ...     Lifestyle   
3  for when a little water damage\nfeels like a m...     

In [14]:
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================
# 1. Load fixed pair dataset
# =========================
pair_df = pd.read_csv("/kaggle/working/stage2_train_pairs_fixed.csv")

print("Original shape:", pair_df.shape)

# Faster test sample first if needed
pair_df = pair_df.sample(n=min(200000, len(pair_df)), random_state=42).reset_index(drop=True)

print("Working shape:", pair_df.shape)

# =========================
# 2. Features and label
# =========================
feature_cols = [
    "stage1_confidence",
    "behaviour_score",
    "text_similarity",
    "category_match",
    "ad_type_targeted"
]

label_col = "label"

pair_df[feature_cols] = pair_df[feature_cols].fillna(0)
pair_df[label_col] = pd.to_numeric(pair_df[label_col], errors="coerce").fillna(0).astype(int)

X = pair_df[feature_cols]
y = pair_df[label_col]

print("\nLabel distribution:")
print(y.value_counts(normalize=True))

# =========================
# 3. Split
# =========================
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Validation shape:", X_val.shape)

# =========================
# 4. Train
# Try WITHOUT class_weight first
# =========================
stage2_model = LogisticRegression(
    max_iter=300,
    random_state=42,
    solver="saga",
    n_jobs=-1
)

stage2_model.fit(X_train, y_train)

# =========================
# 5. Evaluate
# =========================
y_pred = stage2_model.predict(X_val)
y_prob = stage2_model.predict_proba(X_val)[:, 1]

acc = accuracy_score(y_val, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_val,
    y_pred,
    average="weighted",
    zero_division=0
)

print("\n=== Stage 2 Suitability Model Results (Fixed) ===")
print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1-score :", f1)

if len(np.unique(y_val)) > 1:
    print("ROC-AUC  :", roc_auc_score(y_val, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

print("\nDetailed Report:")
print(classification_report(y_val, y_pred, zero_division=0))

coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": stage2_model.coef_[0]
}).sort_values(by="coefficient", ascending=False)

print("\nFeature Coefficients:")
print(coef_df)

joblib.dump(stage2_model, "/kaggle/working/stage2_suitability_model_fixed.pkl")
print("\nSaved: /kaggle/working/stage2_suitability_model_fixed.pkl")

Original shape: (1200000, 13)
Working shape: (200000, 13)

Label distribution:
label
0    0.979285
1    0.020715
Name: proportion, dtype: float64

Train shape: (160000, 5)
Validation shape: (40000, 5)

=== Stage 2 Suitability Model Results (Fixed) ===
Accuracy : 0.979275
Precision: 0.958979525625
Recall   : 0.979275
F1-score : 0.9690210057975773
ROC-AUC  : 0.7585222432131498

Confusion Matrix:
[[39171     0]
 [  829     0]]

Detailed Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     39171
           1       0.00      0.00      0.00       829

    accuracy                           0.98     40000
   macro avg       0.49      0.50      0.49     40000
weighted avg       0.96      0.98      0.97     40000


Feature Coefficients:
             feature  coefficient
3     category_match     3.361123
2    text_similarity     2.563053
1    behaviour_score     1.373224
0  stage1_confidence     0.765251
4   ad_type_targeted    -0.452126


In [15]:
import pandas as pd
import numpy as np
import joblib

# =========================
# 1. Load models and data
# =========================
stage2_model = joblib.load("/kaggle/working/stage2_suitability_model_fixed.pkl")

dev_df = pd.read_csv("/kaggle/working/dev_with_behaviour_score.csv")
ads_df = pd.read_csv("/kaggle/input/datasets/teshanlakruwan/ad-pool/ads_pool.csv")

ads_df["ad_text"] = ads_df["ad_text"].fillna("").astype(str)
ads_df["category"] = ads_df["category"].fillna("Unknown").astype(str)
ads_df["type"] = ads_df["type"].fillna("generic").astype(str)

# =========================
# 2. Helper similarity function
# =========================
def tokenize_text(text):
    if pd.isna(text):
        return []
    return str(text).lower().split()

def simple_overlap_similarity(text1, text2):
    set1 = set(tokenize_text(text1))
    set2 = set(tokenize_text(text2))

    if len(set1) == 0 or len(set2) == 0:
        return 0.0

    return len(set1.intersection(set2)) / len(set1.union(set2))

# =========================
# 3. Build page text if needed
# =========================
if "page_text" not in dev_df.columns:
    if "text" in dev_df.columns:
        dev_df["page_text"] = dev_df["text"].fillna("").astype(str)
    else:
        dev_df["page_text"] = (
            dev_df["title"].fillna("").astype(str) + " " +
            dev_df["abstract"].fillna("").astype(str) + " " +
            dev_df["subcategory"].fillna("").astype(str)
        ).str.strip()

# =========================
# 4. Choose one page
# =========================
page_idx = 0
page_row = dev_df.iloc[page_idx]

page_text = str(page_row["page_text"])
predicted_category = str(page_row["predicted_category"])
stage1_confidence = float(page_row["stage1_confidence"])
behaviour_score = float(page_row["behaviour_score"])

# =========================
# 5. Candidate selection
# =========================
same_cat_n = 5
other_cat_n = 3
generic_n = 2

same_cat_ads = ads_df[ads_df["category"] == predicted_category]
other_cat_ads = ads_df[ads_df["category"] != predicted_category]
generic_ads = ads_df[ads_df["type"].str.lower() == "generic"]

sampled_same = (
    same_cat_ads.sample(n=min(same_cat_n, len(same_cat_ads)), random_state=42)
    if len(same_cat_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
)

sampled_other = (
    other_cat_ads.sample(n=min(other_cat_n, len(other_cat_ads)), random_state=42)
    if len(other_cat_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
)

sampled_generic = (
    generic_ads.sample(n=min(generic_n, len(generic_ads)), random_state=42)
    if len(generic_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
)

candidate_ads = pd.concat(
    [sampled_same, sampled_other, sampled_generic],
    ignore_index=True
).drop_duplicates(subset=["ad_id"]).reset_index(drop=True)

# =========================
# 6. Score ads
# =========================
rank_rows = []

for _, ad_row in candidate_ads.iterrows():
    text_similarity = simple_overlap_similarity(page_text, ad_row["ad_text"])
    category_match = int(predicted_category == ad_row["category"])
    ad_type_targeted = int(str(ad_row["type"]).lower() == "targeted")

    feature_row = pd.DataFrame([{
        "stage1_confidence": stage1_confidence,
        "behaviour_score": behaviour_score,
        "text_similarity": text_similarity,
        "category_match": category_match,
        "ad_type_targeted": ad_type_targeted
    }])

    suitability_score = stage2_model.predict_proba(feature_row)[0][1]

    rank_rows.append({
        "ad_id": ad_row["ad_id"],
        "ad_category": ad_row["category"],
        "ad_type": ad_row["type"],
        "text_similarity": text_similarity,
        "category_match": category_match,
        "suitability_score": suitability_score,
        "ad_text": ad_row["ad_text"]
    })

ranked_ads = pd.DataFrame(rank_rows).sort_values(
    by="suitability_score",
    ascending=False
).reset_index(drop=True)

print("=== NEW PAGE ===")
print("Page index:", page_idx)
print("Predicted category:", predicted_category)
print("Stage 1 confidence:", stage1_confidence)
print("Behaviour score:", behaviour_score)
print("\nPage text:")
print(page_text[:500])

print("\n=== TOP RANKED ADS (FIXED MODEL) ===")
print(ranked_ads.head(10))

# =========================
# 7. Optional fallback rule
# =========================
top_ad = ranked_ads.iloc[0]
threshold = 0.55

if top_ad["suitability_score"] < threshold:
    generic_ranked = ranked_ads[ranked_ads["ad_type"].str.lower() == "generic"]
    final_ad = generic_ranked.iloc[0] if len(generic_ranked) > 0 else top_ad
else:
    final_ad = top_ad

print("\n=== FINAL SELECTED AD ===")
print(final_ad)

=== NEW PAGE ===
Page index: 0
Predicted category: Food_Drink
Stage 1 confidence: 0.9949928190451234
Behaviour score: 0.0088640749931072

Page text:
How much turkey do you need to buy per person? When hosting Thanksgiving dinner, there are a lot of questions regarding the menu that may come up. tipsandtricks

=== TOP RANKED ADS (FIXED MODEL) ===
      ad_id    ad_category   ad_type  text_similarity  category_match  \
0  AD_00343     Food_Drink  targeted         0.051282               1   
1  AD_00775     Food_Drink  targeted         0.027778               1   
2  AD_00844     Food_Drink  targeted         0.023256               1   
3  AD_00394     Food_Drink  targeted         0.022222               1   
4  AD_00809     Food_Drink  targeted         0.000000               1   
5   GEN_002        General   generic         0.028571               0   
6   GEN_005        General   generic         0.000000               0   
7  AD_00595      Lifestyle  targeted         0.000000               